# QM9 3D Coordinate Diffusion: Training & Sampling Demo

Walks through the end-to-end `miniqm` pipeline on real QM9 molecules:
1. Loading pre-collated QM9 batches via `DataSource` and transferring to device with `Batch.to_jax()`.
2. Inspecting Karras EDM preconditioning factors ($c_{\text{in}}, c_{\text{skip}}, c_{\text{out}}, \lambda$) and the Euler rollout noise schedule.
3. Training the `Denoiser` Transformer on a QM9 batch with JIT-compiled `train_step` and `eval_step`.
4. Sampling 3D conformers from Gaussian noise with `sample_coordinates` and comparing Kabsch-aligned structures against QM9 DFT coordinates.

In [ ]:
import functools
import pathlib
from typing import Final

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from miniqm import data_source, geometry, losses, model, noise, sampling, train, viz

DATA_DIR: Final = pathlib.Path("../data").resolve()
VALIDATION_PATH: Final = DATA_DIR / "validation.parquet"

source = data_source.DataSource(VALIDATION_PATH, max_molecules=16)
host_batch = next(source.iter_batches(batch_size=8))
jax_batch = host_batch.to_jax()

print(
    f"atom_features: {jax_batch.atom_features.shape}, "
    f"bond_features: {jax_batch.bond_features.shape}, "
    f"positions: {jax_batch.positions.shape}"
)
print(f"Valid atom counts per molecule: {jnp.sum(jax_batch.atom_mask, axis=-1).tolist()}")

In [ ]:
sigma_grid = jnp.logspace(-3, 2, 200, dtype=jnp.float32)
factors = noise.preconditioning_factors(sigma_grid)
schedule_sigmas = noise.rollout_schedule(40)

fig, (ax_factors, ax_sched) = plt.subplots(1, 2, figsize=(11, 4))

ax_factors.plot(sigma_grid, factors.c_in, label=r"$c_{\mathrm{in}}(\sigma)$")
ax_factors.plot(sigma_grid, factors.c_skip, label=r"$c_{\mathrm{skip}}(\sigma)$")
ax_factors.plot(sigma_grid, factors.c_out, label=r"$c_{\mathrm{out}}(\sigma)$")
ax_factors.set_xscale("log")
ax_factors.set_xlabel(r"Noise level $\sigma$ ($\mathrm{\AA}$)")
ax_factors.set_ylabel("Scaling coefficient")
ax_factors.set_title("Karras EDM Preconditioning Factors")
ax_factors.legend()
ax_factors.grid(alpha=0.3)

ax_sched.plot(np.arange(len(schedule_sigmas)), schedule_sigmas, marker="o", markersize=3)
ax_sched.set_yscale("log")
ax_sched.set_xlabel("Discretization step $i$")
ax_sched.set_ylabel(r"Schedule noise $\sigma_i$ ($\mathrm{\AA}$)")
ax_sched.set_title(r"Karras Rollout Schedule ($\rho = 7$, $N = 40$)")
ax_sched.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
demo_config = train.TrainConfig(
    model=model.DenoiserConfig(
        transformer=model.TransformerConfig(hidden_dim=128, num_layers=4, num_heads=4),
        noise=noise.NoiseConfig(log_sigma_mean=-0.5, log_sigma_std=0.8),
    ),
    loss=losses.LossConfig(distance_loss_weight=0.5),
    learning_rate=1e-3,
    warmup_steps=20,
    decay_steps=300,
    min_learning_rate=1e-4,
    ema_decay=0.98,
)

state = train.init_train_state(jax.random.key(0), jax_batch, config=demo_config)
jit_train_step = jax.jit(functools.partial(train.train_step, config=demo_config))
jit_eval_step = jax.jit(functools.partial(train.eval_step, config=demo_config))

num_steps = 200
eval_rng = jax.random.key(42)
history_total: list[float] = []
history_rmsd: list[float] = []
history_dist: list[float] = []

for step_idx in range(num_steps):
    step_out = jit_train_step(state, jax_batch)
    state = step_out.state
    eval_out = jit_eval_step(state.params, eval_rng, jax_batch)
    history_total.append(float(eval_out.total_loss))
    history_rmsd.append(float(eval_out.rmsd))
    history_dist.append(float(eval_out.distance_loss))
    if (step_idx + 1) % 50 == 0 or step_idx == 0:
        print(
            f"step {step_idx + 1:3d} | total_loss={history_total[-1]:.4f} "
            f"| rmsd={history_rmsd[-1]:.3f} A | dist_loss={history_dist[-1]:.4f} "
            f"| grad_norm={float(step_out.grad_norm):.2f}"
        )

fig, (ax_loss, ax_rmsd) = plt.subplots(1, 2, figsize=(11, 4))
steps_axis = np.arange(1, num_steps + 1)
ax_loss.plot(steps_axis, history_total, label="Total Diffusion Loss")
ax_loss.plot(steps_axis, history_dist, label="Local Pairwise Distance MSE")
ax_loss.set_xlabel("Optimization step")
ax_loss.set_ylabel("Loss")
ax_loss.set_title("Single-Batch Diffusion Loss Convergence")
ax_loss.legend()
ax_loss.grid(alpha=0.3)

ax_rmsd.plot(steps_axis, history_rmsd, color="tab:green")
ax_rmsd.set_xlabel("Optimization step")
ax_rmsd.set_ylabel(r"Kabsch-aligned RMSD ($\mathrm{\AA}$)")
ax_rmsd.set_title("Denoised Coordinate RMSD at Fixed Noise Realization")
ax_rmsd.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
denoiser = model.Denoiser(config=demo_config.model)
sampling_cfg = sampling.SamplingConfig(
    num_steps=40,
    noise=noise.NoiseConfig(s_min=1e-3, s_max=5.0),
)
trajectory = sampling.sample_trajectory(
    jax.random.key(7),
    denoiser,
    state.params,
    batch=jax_batch,
    config=sampling_cfg,
)
generated_positions = trajectory[-1]

aligned_generated = geometry.kabsch_align(
    reference=jax_batch.positions,
    target=generated_positions,
    atom_mask=jax_batch.atom_mask,
)
atom_counts = jnp.sum(jax_batch.atom_mask, axis=-1)
per_mol_rmsd = jnp.sqrt(
    jnp.sum(jnp.square(aligned_generated - jax_batch.positions), axis=(-1, -2)) / atom_counts
)
best_idx = int(jnp.argmin(per_mol_rmsd))
print(f"Sampled conformer RMSDs (A): {[round(float(val), 3) for val in per_mol_rmsd]}")
print(
    f"Best molecule index={best_idx} ({int(atom_counts[best_idx])} atoms), "
    f"Kabsch RMSD = {float(per_mol_rmsd[best_idx]):.3f} A"
)
print("3D Overlay: QM9 DFT Reference (translucent CPK) vs. Sampled Conformer (cyan):")
viz.show_overlay(
    host_batch.positions[best_idx],
    np.asarray(aligned_generated[best_idx]),
    host_batch.atom_features[best_idx],
    host_batch.atom_mask[best_idx],
)

In [ ]:
print(
    f"Animated 3D reverse-diffusion trajectory ({trajectory.shape[0]} frames) "
    f"for molecule {best_idx}:"
)
viz.show_trajectory(
    np.asarray(trajectory[:, best_idx]),
    host_batch.atom_features[best_idx],
    host_batch.atom_mask[best_idx],
    interval_ms=70,
)